# 1. Introducción a la evaluación de LLMs
# ---------------------------------------


En este notebook, exploraremos las métricas de evaluación más comunes.


# 2. Métricas comunes para evaluar LLMs
# ------------------------------------


Las métricas fundamentales para evaluar LLMs incluyen:

1. Exactitud (Accuracy): La exactitud mide la proporción de respuestas o predicciones correctas sobre el total de casos evaluados. En el contexto de LLMs, puede referirse a la capacidad del modelo para producir respuestas que coinciden con las respuestas de referencia (ground truth) en diversos aspectos como el contenido factual, la estructura o el significado semántico.

2. Precisión y Recall: En extracción de información, la precisión mide si la información extraída es correcta, mientras que el recall mide si se extrajo toda la información relevante. En sistemas RAG, la precisión evalúa si el contexto recuperado es relevante, mientras que el recall evalúa si se recuperó toda la información necesaria.

3. Perplexidad: Mide qué tan bien un modelo predice una muestra de texto.

5. Métricas específicas de RAG:
   - Fidelidad (Faithfulness): ¿La respuesta se basa en el contexto proporcionado?
   - Relevancia: ¿La respuesta aborda la pregunta?
   - Precisión del contexto: ¿El contexto recuperado es relevante?
   - Recall del contexto: ¿Se recuperó toda la información relevante?


# 3. Introducción a RAGAS
# -----------------------

RAGAS es una biblioteca para evaluar sistemas de RAG (Retrieval Augmented Generation).

RAGAS utiliza LLMs como evaluadores y, como ya hemos mencionado, está diseñada específicamente para evaluar todos los componentes de un sistema RAG: recuperación (retrieval), 
aumento (augmentation) y generación.

# 4. Métricas de RAGAS en detalle
# ------------------------------


## 4.1 Faithfulness (Fidelidad)

La métrica de fidelidad evalúa si la respuesta generada es fiel al contexto proporcionado o han habido alucinaciones.
Una respuesta con alta fidelidad no contiene información alucinaciones.

### Fórmula:
Faithfulness = Número de afirmaciones respaldadas / Número total de afirmaciones

### Qué hace RAGAS por detrás:
- RAGAS utiliza prompts como "¿Esta afirmación está respaldada por el contexto?"
- Utiliza técnicas de verificación de atribución para detectar hallucinations
- Puntúa cada afirmación como respaldada (1) o no respaldada (0)

## 4.2 Answer Relevancy (Relevancia de la respuesta)

Esta métrica evalúa si la respuesta es relevante para la pregunta formulada.
Una respuesta con alta relevancia nos indica que aborda directamente lo que se preguntó.

### Fórmula:
Answer Relevancy = Puntaje de relevancia asignado por el LLM (0-1)

### Qué hace RAGAS por detrás:
1. Compara la pregunta con la respuesta usando un LLM
2. Evalúa si la respuesta aborda los aspectos clave de la pregunta
3. Considera la coherencia y enfoque de la respuesta



## 4.3 Context Precision (Precisión del contexto)

Mide cuánto del contexto recuperado es realmente relevante para responder la pregunta.
Una baja precisión significa que se proporcionó contexto innecesario o irrelevante.

### Fórmula:
Context Precision = Número de segmentos relevantes / Número total de segmentos

### Qué hace RAGAS por detrás:
1. Divide el contexto en segmentos (normalmente párrafos o chunks)
2. Evalúa la relevancia de cada segmento para la pregunta (similitud semántica entre embeddings)
3. Calcula la proporción de segmentos relevantes



## 4.4 Context Recall (Recuperación del contexto)

Evalúa si el contexto proporcionado contiene toda la información necesaria para responder la pregunta.
Un bajo recall significa que falta información esencial para responder a la pregunta.

### Fórmula:
Context Recall = Información esencial presente en el contexto / Total de información esencial

### Qué hace RAGAS por detrás:
1. Extrae la información clave de la respuesta de referencia (ground truth)
2. Comprueba si esta información está presente en el contexto recuperado (similitud semántica)
3. Calcula la proporción de información esencial que está presente


## 4.5 Context Relevancy (Relevancia del contexto)

Esta métrica evalúa la calidad general del contexto en relación con la pregunta.
Combina aspectos de precisión y recall del contexto.

### Fórmula:
Context Relevancy = Puntuación ponderada de relevancia semántica

### Qué hace RAGAS por detrás:
1. Evalúa la relevancia semántica entre la pregunta y cada fragmento del contexto (similitud semántica entre embeddings)
2. Considera la diversidad y completitud de la información
3. Pondera la relevancia según la importancia para la pregunta






In [ ]:
import sys

In [ ]:
#Librerías que hemos instalado en otros notebooks:
!{sys.executable} -m pip install langchain_openai
!{sys.executable} -m pip install langchain_core
!{sys.executable} -m pip install langchain-community

In [ ]:
!{sys.executable} -m pip install pandas
!{sys.executable} -m pip install matplotlib
!{sys.executable} -m pip install numpy
!{sys.executable} -m pip install ragas

In [ ]:
API_KEY=""

In [ ]:
import os
os.environ["OPENAI_API_KEY"] = API_KEY

In [ ]:
# Evaluación de Modelos de Lenguaje (LLMs) con RAGAS
# ----------------------------------------------

# Este notebook introduce las métricas clave para evaluar Modelos de Lenguaje de Gran Escala (LLMs)
# y muestra cómo implementar evaluaciones usando la biblioteca RAGAS.

# Importamos las bibliotecas necesarias
from langchain_openai import ChatOpenAI
import pandas as pd
import numpy as np
import json
import matplotlib.pyplot as plt
from ragas import evaluate, EvaluationDataset
from ragas.llms import LangchainLLMWrapper
from ragas.metrics import LLMContextRecall, Faithfulness,LLMContextPrecisionWithReference, ResponseRelevancy

# Preparación de datos para evaluación
# --------------------------------------

# Creamos un conjunto de datos de ejemplo para evaluar un sistema RAG
with open("data_eval_ragas.json", "rb") as f:
        eval_doc = json.load(f)

dataset=EvaluationDataset.from_dict(eval_doc)


evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0, api_key=API_KEY)) #Añade aqui el api_key de OpenAI

# Evaluación con RAGAS
# ----------------------------------------
result = evaluate(
        dataset=dataset,
        metrics=[LLMContextRecall(), Faithfulness(),LLMContextPrecisionWithReference(),ResponseRelevancy()],
        llm=evaluator_llm,
    )

df=result.to_pandas()



In [ ]:
df.to_csv("evaluation_result.csv", index=False)

In [ ]:
df

In [ ]:

# Visualización de resultados
# -----------------------------

def plot_evaluation_results(df):
    metrics_columns = df.columns[4:]
    
    results = {}
    for metric in metrics_columns:
        results[metric] = df[metric].mean()
    
    metrics = list(results.keys())
    scores = list(results.values())
    
    plt.figure(figsize=(12, 6))
    bars = plt.bar(metrics, scores, color=['#3498db', '#2ecc71', '#e74c3c', 
                                           '#f39c12', '#9b59b6', '#1abc9c'])
    
    plt.ylim(0, 1)
    plt.title('Evaluación de LLM con métricas RAGAS')
    plt.ylabel('Puntuación')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()
    
    return results


def plot_metric_by_question(df, metric):
    questions = df['user_input'].tolist()
    short_questions = [q[:40] + "..." if len(q) > 40 else q for q in questions]
    
    plt.figure(figsize=(10, 6))
    bars = plt.bar(short_questions, df[metric], color='#3498db')
    
    plt.ylim(0, 1)
    plt.title(f'Comparación de {metric} por pregunta')
    plt.ylabel('Puntuación')
    plt.xticks(rotation=45, ha='right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    
    for bar in bars:
        height = bar.get_height()
        plt.text(bar.get_x() + bar.get_width()/2., height + 0.01,
                 f'{height:.2f}', ha='center', va='bottom')
    
    plt.tight_layout()
    plt.show()


def plot_scores_by_question(df):
    metrics = df.columns[4:]
    questions = df['user_input'].tolist()
    short_questions = [q[:30] + "..." if len(q) > 30 else q for q in questions]
    
    x = np.arange(len(short_questions))
    width = 0.8 / len(metrics)
    colors = ['#3498db', '#2ecc71', '#e74c3c', '#f39c12', '#9b59b6', '#1abc9c']
    
    plt.figure(figsize=(14, 6))
    for i, metric in enumerate(metrics):
        offset = (i - len(metrics) / 2 + 0.5) * width
        bars = plt.bar(x + offset, df[metric], width, label=metric, color=colors[i % len(colors)])
    
    plt.ylim(0, 1.1)
    plt.title('Puntuaciones por pregunta y métrica')
    plt.ylabel('Puntuación')
    plt.xticks(x, short_questions, rotation=45, ha='right')
    plt.legend(loc='upper right')
    plt.grid(axis='y', linestyle='--', alpha=0.7)
    plt.tight_layout()
    plt.show()


def plot_metrics_correlation(df):
    metrics = df.columns[4:]
    
    corr = df[metrics].corr()
    
    plt.figure(figsize=(10, 8))
    plt.imshow(corr, cmap='coolwarm', vmin=-1, vmax=1)
    
    plt.colorbar(label='Correlación')
    plt.xticks(np.arange(len(metrics)), metrics, rotation=45, ha='right')
    plt.yticks(np.arange(len(metrics)), metrics)
    
    for i in range(len(metrics)):
        for j in range(len(metrics)):
            plt.text(j, i, f'{corr.iloc[i, j]:.2f}', 
                     ha='center', va='center', color='black')
    
    plt.title('Correlación entre métricas RAGAS')
    plt.tight_layout()
    plt.show()


In [ ]:
# Visualizar la correlación entre métricas
if len(df) >= 3:  # Necesitamos al menos 3 filas para una correlación significativa
    plot_metrics_correlation(df)
# Ejemplo: visualizar una métrica 
plot_metric_by_question(df, df.columns[5])
plot_metric_by_question(df, df.columns[6])
plot_metric_by_question(df, df.columns[7])
# Visualizamos las puntuaciones por pregunta
# Visualizamos los resultados agregados
results = plot_evaluation_results(df)
print("Resultados promedio por métrica:")
for metric, score in results.items():
    print(f"{metric}: {score:.4f}")

In [ ]:
from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import CharacterTextSplitter, RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_classic.chains.combine_documents import create_stuff_documents_chain
from langchain_classic.chains import create_retrieval_chain


# 1. Cargar el documento
loader = TextLoader("textos.txt")
documents = loader.load()

# 2. Dividir el texto en chunks
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size=300,
    chunk_overlap=100,
    separators=["\n\n", "\n", ". ", " ", ""]  # Prioriza divisiones naturales
)
texts = text_splitter.split_documents(documents)

# 3. Crear embeddings y almacenarlos en una base de datos vectorial
embeddings = OpenAIEmbeddings(api_key=API_KEY)
vectorstore = FAISS.from_documents(texts, embeddings)

# 4. Crear un modelo de lenguaje
llm = ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0.5, api_key=API_KEY)

# 5. Crear una plantilla de prompt
# create_stuff_documents_chain espera {context} (lista de documentos) y {input}
prompt = ChatPromptTemplate.from_template("""Usa la siguiente información para responder la pregunta del usuario. Si no sabes la respuesta, inventate algo relacionado con el topic.

Contexto: {context}

Pregunta: {input}

Respuesta:""")

# 6. Crear la cadena de RAG
combine_docs_chain = create_stuff_documents_chain(llm, prompt)
qa_chain = create_retrieval_chain(
    vectorstore.as_retriever(),
    combine_docs_chain,
)


In [ ]:
## Cargamos el json
with open("qa_textos.json", "rb") as f:
        texto_qa = json.load(f)

In [ ]:
for input in texto_qa:
    contexts = []
    question = input["user_input"]
    reference = input["reference"]

    result = qa_chain.invoke({"input": question})
    input["response"] = result["answer"]

    for context in result["context"]:
        contexts.append(context.page_content)
    input["retrieved_contexts"] = contexts


In [ ]:
dataset=EvaluationDataset.from_list(texto_qa)

In [ ]:
dataset=EvaluationDataset.from_list(texto_qa)

evaluator_llm = LangchainLLMWrapper(ChatOpenAI(model_name="gpt-3.5-turbo", temperature=0, api_key=API_KEY))

# Evaluación con RAGAS
# ----------------------------------------
result = evaluate(
        dataset=dataset,
        metrics=[LLMContextRecall(), Faithfulness(),LLMContextPrecisionWithReference(),ResponseRelevancy()],
        llm=evaluator_llm,
    )

df=result.to_pandas()

In [ ]:
df.describe()

In [ ]:
# Visualizar la correlación entre métricas
if len(df) >= 3:  # Necesitamos al menos 3 filas para una correlación significativa
    plot_metrics_correlation(df)
# Ejemplo: visualizar una métrica 
plot_metric_by_question(df, df.columns[5])
# Visualizamos las puntuaciones por pregunta
plot_scores_by_question(df)
# Visualizamos los resultados agregados
results = plot_evaluation_results(df)
print("Resultados promedio por métrica:")
for metric, score in results.items():
    print(f"{metric}: {score:.4f}")

## Ejercicio 1: Modifica tu prompt

- Objetivo: Evaluar cómo diferentes componentes y configuraciones de un sistema RAG afectan su rendimiento.
- Pasos:
  1. Una vez evaluado nuestro sistema RAG, identifica las zonas de mejora
  2. Con la finalidad de obtener mejoras en las métricas haz 3 modificaciones. Una de prompt, otra de chunking, y otra en la llm.
  3. ¿Con cuál has obtenido mejores resultados?